In [1]:
# move to main dir and import library
import os
os.chdir('../')
from library import *

# filter the warnings for clarity
import warnings
warnings.filterwarnings("ignore")

## Jupyter Notebook - ECL Update ⏰⚙️💾

Welcome to the ECL dataset update notebook! This comprehensive notebook is designed to streamline the process of updating the dataset, ensuring that the most recent 10K filings are included. In today's fast-paced financial landscape, having up-to-date and reliable data is crucial for making informed decisions. This notebook provides you with a systematic approach to update your dataset by crawling and parsing new textual data, matching it with CompuStat records, and saving the new version of the dataset on your machine. We will be taking the following steps:

**Step 1 - Crawl and parse new textual data:** 
In the first step op of the update process, we will make use of the EDGAR-crawler open-source GitHub repo (https://github.com/nlpaueb/edgar-crawler) to extract and parse the textual data from the most recent 10K filings. We have added the repo as a submodule such that it is smoothly integrated with our code.

**Step 2 - Match the crawled records with CompuStat records:**
Once you have obtained the textual data of the new 10K filings, the notebook provides code to match that data with CompuStat records containing the corresponding numerical financial data. *Hence, this step requires access to CompuStat* via (1) the WRDS Python API or (2) a local copy of CompuStat. We perform the matching by using the CIK, fiscal year end and company name in a variety of ways to achieve optimal alignment of both data sources.

**Step 3 - Save new version of the dataset:**
After successfully integrating the textual data and numerical data, the notebook allows you to save a fresh version of the updated dataset on your local machine. This ensures that you have a clean and up-to-date multimodal dataset ready for analysis, modeling, or any other financial tasks of interest.

```Note that the bankruptcy labels are NOT updated. The source dataset, LoPucki BRD, is no longer updated.```

### Set the parameters

Before we start, assign the desired values to the parameters in the cell below to ensure a smooth execution of the update process. 
- Specify the desired date range within which you wish to collect additional data. *Note that data has been collected until May 2023 (there is no need to collect data prior to this date).*
- Provide an email address that will be declared to the U.S. Securities and Exchange Commission (SEC). This is needed as we will collect data from the Electronic Data Gathering, Analysis, and Retrieval (EDGAR) website and their servers. 
- Specify the path to the EDGAR corpus. We will add the new records to this dataset during the update process.
- Specify the path to your local copy of CompuStat. If you are working with the WRDS Python API, this can be left blank.

For demonstration purposes, we update the dataset with a small number of records. The variable *demo* can be set to False when updating your version of the dataset.

In [3]:
start_year = 2023 
end_year = 2024
user_agent = ''
wrds_username = ''
path_corpus = os.getcwd() + '/data/'
path_compustat = os.getcwd() + '/data/CompuStat/data.csv'

# Set demo to False when updating the dataset
demo = True
if demo:
    start_year = 2022
    end_year = 2022

### Step 1: Crawl and parse new texual data

As mentioned earlier, we have integrated the EDGAR-crawler GitHub repository as a submodule in our project. This enables us to seamlessly access all the code from this repository via the designated subfolder *'./edgar-crawler'*. To initiate the update process, we begin by adjusting the *config.json* file. This file contains important configurable parameters, including the specific types of reports to crawl (such as 10K, 10KSB, 10K405, and more), the desired date range for data collection, and the specific items to parse from the reports, among others. By updating this file with the *update_config_file()* function, we ensure that the crawler is ready to gather the most relevant and accurate information for our dataset.

In [4]:
# The path to the config file.
# If the edgar-crawler code is added as a submodule, this should not be changed.
path_config = './update/edgar-crawler/config.json'

# Update the config file and store a copy.
copy_config = update_config_file(path_config, start_year, end_year, user_agent, demo)

Moving forward, we execute the code for (1) crawling the 10K filings and (2) parsing the relevant textual items from each report. As the code runs, the resulting data will be stored in the *'./edgar-crawler/dataset'* folder on your local machine. Each document will be saved as a JSON file, which we will process further in the next stage.

**Note: the cell below might take some time to run. If you run the python scripts in a separate console, a progress bar will show.**

In [5]:
%%capture
# Ignore the output of the cell.

# Change the working directory to the GitHub submodule.
os.chdir('./update/edgar-crawler/')

# Run the crawling/parsing code. This might take a while!
! python edgar_crawler.py
! python extract_items.py

# Move back to main directory.
os.chdir('../..')

# Reset the config file.
with open(os.getcwd() + '/update/edgar-crawler/config.json', 'w') as fp:    
    json.dump(copy_config,fp)

In this part of the notebook, we will move the processed documents to the correct place in the EDGAR-corpus and clean up the */update/* folder. Additionally, as part of this process, we store the meta-data of each document into a Pandas dataframe. This meta-data allow us to link the processed textual records to CompuStat records (containing corresponding numerical/categorical data) in the next stage.

In [6]:
# The path to the newly extracted records.
# If the edgar-crawler code is added as a submodule, this should not be changed.
path_records = os.getcwd() + '/update/edgar-crawler/datasets/EXTRACTED_FILINGS/'

# Move the records from the update folder to the EDGAR-corpus and store meta-data.
meta_data = move_extracted_filings(path_records, path_corpus, demo)

In [7]:
# Let's inspect some records!
meta_data.head()

,cik,company,filing_type,filing_date,period_of_report,state_of_inc,state_location,sic,filename
0,1018724,AMAZON COM INC,10-K,2022-02-04,2021-12-31,DE,WA,5961,/2021/1018724_10K_2021_0001018724-22-000005.json
1,1318605,"Tesla, Inc.",10-K,2022-02-07,2021-12-31,DE,CA,3711,/2021/1318605_10K_2021_0000950170-22-000796.json
2,22701,COMMUNICATIONS SYSTEMS INC,10-K,2022-03-14,2021-12-31,MN,MN,3661,/2021/22701_10K_2021_0000022701-22-000004.json
3,77476,PEPSICO INC,10-K,2022-02-10,2021-12-25,NC,NY,2080,/2021/77476_10K_2021_0000077476-22-000010.json


### Step 2: Match the crawled records with CompuStat records

In this part of the notebook, we will link the textual data from the 10K filings to CompuStat records containing the corresponding numerical and categorical data. As a first step, we fetch all relevant CompuStat data using (1) the WRDS Python API or (2) a local copy of CompuStat. 

When using **the WRDS Python API** for the first time, you will need to login using your WRDS username and password. For a thorough explanation on how to install and use the API, see the documentation: https://wrds-www.wharton.upenn.edu/pages/support/programming-wrds/programming-python/querying-wrds-data-python/.

If you are working with **a local copy of CompuStat** and do not have access to a WRDS account, you can update the dataset anyway! Make sure that the CompuStat version you are working with is up-to-date and contains the most recent records. We have stored our version (the *data.csv* file) in the *'./data/CompuStat/'* folder but you can store this anywhere you like, just be sure to change the path at the beginning of the notebook!

#### Option 1: Access CompuStat via the WRDS Python API

In [ ]:
# login the WRDS Python API.
db = wrds.Connection(wrds_username=wrds_username)

# Create the query.
query = """
SELECT gvkey, datadate, cik, conm, at FROM comp_na_annual_all.funda 
WHERE indfmt = 'INDL'
AND datafmt = 'STD'
AND consol = 'C'
AND popsrc = 'D'
AND datadate BETWEEN '1993-01-01' AND '-01-01'
"""
query = query[:185] + str(end_year+1) + query[185:]


# Fetch the data.
comp = db.raw_sql(query)

#### Option 2: Access CompuStat via a local copy

In [8]:
# Get the CompuStat data - might take a while!
comp = get_CompuStat_local(path_compustat, dataset = None, update = True)
# Retain relevant variables
comp = comp[['gvkey', 'datadate', 'cik', 'conm']]

Dropped 115373 rows from CompuStat based on screening variables


#### Matching the records

Now, we match the textual records and the CompuStat records based on two conditions. They have (1) the same company identifier (*cik*) and (2) a fiscal year end (*period_of_report*) that lies within 7 days of eachother. Our analysis revealed that the fiscal year end as reported in CompuStat can be off by a couple of days. This step ensures that the textual data and the numeric data coming from the same 10K filing is matched. Next, we link remaining textual records and CompuStat records that (1) have the same company name (company and conm) and (2) the same fiscal year end. This makes sure that we also link the data for companies where the CIK is missing in CompuStat.

In [9]:
# Cast the variables that will be used in the merge to the correct type.
comp['period_of_report'] = pd.to_datetime(comp['datadate'])
meta_data['period_of_report'] = pd.to_datetime(meta_data['period_of_report'])
comp['gvkey'] = comp['gvkey'].fillna('999999').astype(int)
comp['cik'] = comp['cik'].fillna('999999').astype(int)
meta_data['cik'] = meta_data['cik'].fillna('0000000').astype('int')
comp['conm'] = comp['conm'].str.upper()
meta_data['company'] = meta_data['company'].str.upper()

In [10]:
# Match textual and CompuStat records on CIK and fiscal year end.
update = pd.merge(left=comp, right=meta_data, how='outer', on=['cik', 'period_of_report'], indicator=True)
update = fuzzy_match_period_of_report(dataset=update, edgar=meta_data, compustat=comp, offset=7)

# Match textual and CompuStat records on company name and fiscal year end.
update = fuzzy_match_name(dataset=update, edgar=meta_data, compustat=comp)

# Store only the matched records.
new_records = update.loc[update['_merge'] == 'both']

# Let's inspect some records!
new_records.reset_index(drop=True).head()

,gvkey,datadate,cik,conm,period_of_report,company,filing_type,filing_date,state_of_inc,state_location,sic,filename,_merge
0,64768.0,2021-12-31,1018724,AMAZON.COM INC,2021-12-31,AMAZON COM INC,10-K,2022-02-04,DE,WA,5961,/2021/1018724_10K_2021_0001018724-22-000005.json,both
1,184996.0,2021-12-31,1318605,TESLA INC,2021-12-31,"TESLA, INC.",10-K,2022-02-07,DE,CA,3711,/2021/1318605_10K_2021_0000950170-22-000796.json,both
2,8479.0,2021-12-31,77476,PEPSICO INC,2021-12-25,PEPSICO INC,10-K,2022-02-10,NC,NY,2080,/2021/77476_10K_2021_0000077476-22-000010.json,both
3,3275.0,2021-12-31,22701,COMMUNICATIONS SYSTEMS INC,2021-12-31,COMMUNICATIONS SYSTEMS INC,10-K,2022-03-14,MN,MN,3661,/2021/22701_10K_2021_0000022701-22-000004.json,both


The following two examples show that matching the records in different ways really helps to improve the quality of the dataset. The PEPSICO records (textual and CompuStat) are matched even though they have a fiscal year end that lies 6 days from eachother (i.e. a data quality issue in one of the source datasets). The COMMUNICATIONS SYSTEMS INC records are still matched even though the CIK is missing in CompuStat.

In [11]:
# PEPSICO example.
new_records.loc[new_records['cik'] == 77476, ['company', 'period_of_report', 'datadate']].iloc[0]

company                     PEPSICO INC
period_of_report    2021-12-25 00:00:00
datadate            2021-12-31 00:00:00
Name: 0, dtype: object

In [12]:
# COMMUNICATIONS SYSTEMS INC example.
comp.loc[(comp['gvkey'] == 3275), ['conm', 'period_of_report', 'cik']].iloc[-1]

conm                COMMUNICATIONS SYSTEMS INC
period_of_report           2021-12-31 00:00:00
cik                                     999999
Name: 10959, dtype: object

### Step 3: Save new version of the dataset

In this final step, we append the new records to the dataset. Note that the variables related to the bankruptcy prediction task are set to missing. This adjustment is necessary because, for all records after the year 2023, we do not know the labels of the task (the LoPucki BRD was discontinued on 31/12/2022). The resulting .csv file, containing the most up-to-date data is stored in the same folder as the original dataset with '-updated' appended to the filename.

In [13]:
# Read in the original dataset.
dataset = pd.read_csv('ECL.csv', index_col=0, low_memory=False).drop_duplicates()
nrows = len(dataset)

# Adjust the datatypes.
dataset['datadate'] = pd.to_datetime(dataset['datadate'], dayfirst=True)
dataset['period_of_report'] = pd.to_datetime(dataset['period_of_report'])

In [14]:
# First check if we have any records to add.
if len(new_records!=0):
    
    # All new records are out-of-scope for the bankruptcy prediction task.
    # We set the relevant variables accordingly.
    new_records.loc[:,'can_label'] = ~new_records['filing_date'].isna()
    new_records.loc[:,'qualified'] = False
    new_records.loc[:,'label'] = False
    new_records.loc[:,'bankruptcy_prediction_split'] = 'out-of-scope'
    new_records.loc[:,'bankruptcy_date_1'] = np.NaN
    new_records.loc[:,'bankruptcy_date_2'] = np.NaN
    new_records.loc[:,'bankruptcy_date_3'] = np.NaN
    
    # We store only the relevant columns.
    new_records = new_records[dataset.columns]
    
    # Append the records to the dataset and drop potential duplicates.
    dataset = pd.concat([dataset, new_records])
    dataset = dataset.drop(dataset.loc[dataset[['gvkey', 'filename']].duplicated()].index).reset_index(drop=True)
        
    # Store the result.
    if not demo:
        print('Added ' + str(len(dataset) - nrows) + ' new records')
        dataset.to_csv('ECL-updated.csv')

In [15]:
# Let's inspect some records!
dataset.sample(5)

,cik,company,period_of_report,gvkey,datadate,filename,can_label,qualified,label,bankruptcy_prediction_split,bankruptcy_date_1,bankruptcy_date_2,bankruptcy_date_3,filing_date
78516,885639.0,KOHLS CORP,2021-01-30,25283.0,2021-01-31,/2021/885639_10K_2021_0001564590-21-014145.json,True,Yes,False,test,NaN,NaN,NaN,2021-03-18
116801,1024441.0,FACTORY CARD OUTLET CORP,2000-01-29,64154.0,2000-01-31,/2000/1024441_10K_2000_0000909518-00-000345.json,True,No,False,out-of-scope,NaN,NaN,NaN,2000-05-16
112650,1010470.0,PROVIDENT FINANCIAL HOLDINGS INC,2005-06-30,63178.0,2005-06-30,/2005/1010470_10K_2005_0000939057-05-000266.json,True,Yes,False,train,NaN,NaN,NaN,2005-09-13
91952,892653.0,SPORT HALEY INC,2000-06-30,29997.0,2000-06-30,/2000/892653_10K_2000_0000912057-00-047240.json,True,No,False,out-of-scope,NaN,NaN,NaN,2000-11-06
102649,1910139.0,MOBILEYE GLOBAL INC.,2022-12-31,41640.0,2022-12-31,/2022/1910139_10K_2022_0001104659-23-030588.json,True,out-of-period,False,out-of-scope,NaN,NaN,NaN,2023-03-09
